# Experiment Tracking with MLflow

Track and compare ML experiments using MLflow.

## 1. Install MLflow

In [1]:
# Run once in terminal:
# pip install mlflow

## 2. Imports

In [2]:
from pathlib import Path
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
import matplotlib.pyplot as plt


## 3. Load Dataset

In [3]:
PROJECT_ROOT=Path.cwd().parent
df=pd.read_csv(PROJECT_ROOT/'data'/'raw'/'heart.csv')
df['target']=(df['num']>0).astype(int)
df.drop(columns=['num'],inplace=True)
imp=SimpleImputer(strategy='median')
df[['ca','thal']]=imp.fit_transform(df[['ca','thal']])
X=df.drop(columns=['target'])
y=df['target']
cont=['age','trestbps','chol','thalach','oldpeak']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
sc=StandardScaler()
X_train[cont]=sc.fit_transform(X_train[cont])
X_test[cont]=sc.transform(X_test[cont])


## 4. Configure MLflow

In [4]:
# ==========================================
# Configure MLflow (MLflow 3.x Compatible)
# ==========================================

import mlflow

# Reset tracking URI if already configured
mlflow.set_tracking_uri("")

# Use SQLite backend
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Create / Use experiment
mlflow.set_experiment("Heart Disease Prediction")

print("Tracking URI:", mlflow.get_tracking_uri())

2026/07/12 22:38:25 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/12 22:38:25 INFO mlflow.store.db.utils: Updating database tables
2026/07/12 22:38:55 INFO mlflow.tracking.fluent: Experiment with name 'Heart Disease Prediction' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db


## 5. Logistic Regression Experiment

In [10]:
with mlflow.start_run(run_name="Logistic Regression"):
    model=LogisticRegression(max_iter=5000,solver="liblinear",random_state=42)
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    prob=model.predict_proba(X_test)[:,1]

    metrics={
        "accuracy":accuracy_score(y_test,pred),
        "precision":precision_score(y_test,pred),
        "recall":recall_score(y_test,pred),
        "f1":f1_score(y_test,pred),
        "roc_auc":roc_auc_score(y_test,prob)
    }
    mlflow.log_params({"model":"LogisticRegression","solver":"liblinear","max_iter":5000})
    mlflow.log_metrics(metrics)

    ConfusionMatrixDisplay.from_predictions(y_test,pred)
    ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
    ARTIFACT_DIR.mkdir(exist_ok=True)
    plt.savefig(ARTIFACT_DIR / "lr_confusion.png", bbox_inches="tight")
    plt.close()
    RocCurveDisplay.from_predictions(y_test,prob)
    plt.savefig(ARTIFACT_DIR / "lr_roc.png", bbox_inches="tight")
    plt.close()

    mlflow.log_artifact("lr_confusion.png")
    mlflow.log_artifact("lr_roc.png")
    mlflow.sklearn.log_model(model, name="model")


## 6. Random Forest Experiment

In [11]:
with mlflow.start_run(run_name="Random Forest"):
    model=RandomForestClassifier(n_estimators=200,random_state=42)
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    prob=model.predict_proba(X_test)[:,1]

    metrics={
        "accuracy":accuracy_score(y_test,pred),
        "precision":precision_score(y_test,pred),
        "recall":recall_score(y_test,pred),
        "f1":f1_score(y_test,pred),
        "roc_auc":roc_auc_score(y_test,prob)
    }
    mlflow.log_params({"model":"RandomForest","n_estimators":200})
    mlflow.log_metrics(metrics)

    ConfusionMatrixDisplay.from_predictions(y_test,pred)
    plt.savefig(ARTIFACT_DIR / "rf_confusion.png", bbox_inches="tight")
    plt.close()
    RocCurveDisplay.from_predictions(y_test,prob)
    plt.savefig(ARTIFACT_DIR / "rf_roc.png", bbox_inches="tight")
    plt.close()

    mlflow.log_artifact("rf_confusion.png")
    mlflow.log_artifact("rf_roc.png")
    mlflow.sklearn.log_model(model, name="model")


## 7. Observation

Open the MLflow UI (`mlflow ui`) and compare both runs based on metrics and artifacts.